# Assessing the cheaseBS box-edge runs

Reads back every run already in `runs/`, grouped by discharge and by solver
tolerance, and asks one physics question: **did scaling the profiles actually
move the equilibrium, and did it move the way it should?**

The run notebook (`reshape_convergence.ipynb`) produces; this one only reads, so
nothing here costs solver time.

**What is and is not on this machine.** Each run directory keeps its summary
JSON, which is committed. The per-point solve artifacts -- EQDSKs, profiles,
`iteration_errors.png` -- are gitignored and exist only where the solves ran.
Every table below works anywhere; the `DischargePhysics` overlays need that
machine.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import pathlib, sys
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pedestal_scan.py").exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "reshape_convergence"))

from pedestal_scan import Campaign
import assess_helpers as ah

runs = ah.load_runs()
print(f"{len(runs)} runs, shots {sorted({r['shot'] for r in runs})}, "
      f"tolerances {sorted({r['tol_q'] for r in runs})}")
ah.inventory(runs)

## What was run

`iters` is the four points in order: `Te` 0.70, `Te` 1.30, `ne` 0.70, `ne` 1.30.
`capped` counts points that exhausted `max_iter` -- those did not converge, they
ran out. `has_deltas` marks runs solved after the whole-profile diagnostics were
added, so older runs are missing `dq_max` / `dp_max`.

In [ ]:
df = ah.frame(runs)
print(f"{len(df)} solves total")
ah.artifacts_available(runs)

## Per discharge and tolerance

Each discharge's four box-edge points, grouped by the tolerance they were solved
at. 132543 is the only shot solved at more than one tolerance.

In [ ]:
for shot in sorted(df["shot"].unique()):
    for tol in sorted(df[df.shot == shot]["tol_q"].unique()):
        sub = df[(df.shot == shot) & (df.tol_q == tol)]
        print(f"\n=== {shot}   tol_q = {tol:g}   ({sub['run'].nunique()} run(s)) ===")
        show = sub[["run", "axis", "scale", "d_ped_top", "iters", "capped",
                    "converged", "accepted", "ip_err", "q_err_x0",
                    "dq_max", "dp_max", "wall_s"]]
        display(show.style.format(
            {"d_ped_top": "{:+.1%}", "ip_err": "{:.3%}", "q_err_x0": "{:.2%}",
             "dq_max": "{:.2%}", "dp_max": "{:.2%}", "wall_s": "{:.0f}"},
            na_rep="--").hide(axis="index"))

## Iteration count: what actually sets it

`bs_change` never gates -- it sits below $10^{-4}$ on every point. `q_change`
does. Only the points whose pressure change perturbs $q$ enough to exceed
`tol_q` keep iterating, which is why the same 132543 point costs 18 iterations
at $10^{-4}$ and 2 at $10^{-3}$ **and lands in the same place**.

In [ ]:
piv = df.pivot_table(index=["shot", "tol_q"], columns=["var", "scale"],
                     values="iters", aggfunc="max", dropna=False)
display(piv)

ok = df[df.iters.notna()]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for d, m, c in (("down", "o", "tab:blue"), ("up", "^", "tab:red")):
    s = ok[ok.dir == d]
    ax[0].scatter(s.d_ped_top.abs() * 100, s.iters, marker=m, c=c, s=70,
                  label=f"scale {d}", zorder=3)
ax[0].set_xlabel("|measured pedestal-top change| (%)")
ax[0].set_ylabel("iterations")
ax[0].set_title("magnitude does not order the cost")
ax[0].legend(); ax[0].grid(alpha=.3)

for d, c in (("down", "tab:blue"), ("up", "tab:red")):
    s = ok[ok.dir == d]
    ax[1].scatter(s.ip_err * 100, s.iters, c=c, s=70, label=f"scale {d}", zorder=3)
ax[1].axvline(1.5, ls="--", c="k", lw=1)
ax[1].set_xlabel("final $I_p$ error (%)"); ax[1].set_ylabel("iterations")
ax[1].set_title("dashed = standing 1.5% offset")
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout()

## Did the equilibrium move? The whole-profile deltas

`dq_max` / `dp_max` compare the reconstructed EQDSK against the source over the
**whole** profile, with the radius where the change peaks. This is the direct
test -- cheaseBS reshapes all of it, so a number read at two analysis radii
cannot answer the question.

Watch for two things: whether different perturbations give the *same* delta
(which would mean the number is a reconstruction offset, not a response), and
whether `dp_at` lands in the pedestal, where the axes act, or in the core.

In [ ]:
d = df[df.dq_max.notna()]
if len(d) == 0:
    print("no run has whole-profile deltas yet -- re-solve to populate them")
else:
    display(d[["shot", "tol_q", "axis", "scale", "d_ped_top",
               "dq_max", "dq_at", "dp_max", "dp_at", "ip_err"]].style.format(
        {"d_ped_top": "{:+.1%}", "dq_max": "{:.3%}", "dq_at": "{:.3f}",
         "dp_max": "{:.3%}", "dp_at": "{:.3f}", "ip_err": "{:.3%}"},
        na_rep="--").hide(axis="index"))
    print("\nIf two rows with different perturbations share dq_max/dp_max to "
          "several digits, neither moved the equilibrium -- what is being "
          "measured is the reconstruction's own offset, not a response.")

## Does $I_p$ track the pressure change?

With `qspec=on` CHEASE imposes $q$ and $I_p$ is an output, responding to the
profiles only through the bootstrap current. So $I_p$ error against the source
target is a proxy for whether the reshape reached the current profile at all.

A physical response is monotonic in the pedestal-top change and roughly
symmetric about it. Anything one-sided is a finding.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for shot, m in zip(sorted(df.shot.unique()), ["o", "s", "^", "D"]):
    s = df[(df.shot == shot) & df.ip_err.notna()]
    for var, open_face in (("Te", True), ("ne", False)):
        t = s[s["var"] == var]
        if open_face:
            ax.scatter(t.d_ped_top * 100, t.ip_err * 100, marker=m, s=80,
                       facecolors="none", edgecolors="C0", zorder=3,
                       label=f"{shot} Te")
        else:
            ax.scatter(t.d_ped_top * 100, t.ip_err * 100, marker=m, s=80,
                       c="C3", zorder=3, label=f"{shot} ne")
ax.axhline(1.5, ls="--", c="k", lw=1)
ax.axvline(0, ls=":", c="grey", lw=1)
ax.set_xlabel("measured pedestal-top change (%)")
ax.set_ylabel("final $I_p$ error (%)")
ax.set_title("$I_p$ response to the reshape (open = Te, filled = ne)")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
fig.tight_layout()
print("Points on the dashed line did not move the current profile.")

## Profiles and equilibrium overlaid

`DischargePhysics.plot()` draws geometry, $T$ and $n$ profiles, and the
$q$, $F$, $p$, $p'$, $FF'$ panels in one figure, so a scaling change and the
equilibrium change it caused are read together.

Split by axis on purpose: `Te` and `ne` separately is three curves rather than
five, and within a scan the source is the control. **The `Te` scan should leave
$n_e$ untouched and the `ne` scan should leave $T_e$ untouched** -- if either
moves, the transform is not doing what the axis name says.

In [ ]:
RUN = runs[-1]          # newest; pick another with runs[i] or by tag
camp = Campaign([RUN["shot"]])
print(RUN["tag"], "shot", RUN["shot"], "tol_q", RUN["tol_q"])
fig = ah.overlay(camp, RUN, var="Te")

In [ ]:
fig = ah.overlay(camp, RUN, var="ne")

In [ ]:
# All four at once, for the equilibrium panels where the spread is the point.
fig = ah.overlay(camp, RUN)

## Cross-tolerance check on 132543

The same discharge solved at $10^{-4}$ and $10^{-3}$. If the two tolerances give
the same equilibrium, the extra iterations bought nothing and `tol_q` is a cost
knob rather than a quality knob.

In [ ]:
t = df[df.shot == 132543].sort_values(["tol_q", "axis", "scale"])
display(t[["run", "tol_q", "axis", "scale", "iters", "ip_err", "q_err_x0",
           "wall_s"]].style.format(
    {"ip_err": "{:.4%}", "q_err_x0": "{:.3%}", "wall_s": "{:.0f}"},
    na_rep="--").hide(axis="index"))

print("\nSame points, two tolerances:")
display(t.pivot_table(index=["axis", "scale"], columns="tol_q",
                      values=["iters", "ip_err", "q_err_x0"], aggfunc="first"))

## Assessment

Fill in against the tables above. The questions worth answering:

1. **Did any point move the equilibrium in a way that makes physical sense?**
   A real response is localized where the axis acts (the pedestal), monotonic in
   the pedestal-top change, and roughly symmetric in sign.
2. **Which points are indistinguishable from doing nothing?** Those sitting at
   the standing $1.5\%$ $I_p$ offset with the reconstruction's common
   `dq_max` / `dp_max`.
3. **Is the one-sidedness physical or numerical?** Raising pedestal pressure
   raises the bootstrap current, so *some* asymmetry is expected -- but a
   $-25\%$ density change producing no current response at all is not.
4. **What does 129038 need?** All four points capped and gate-rejected on $I_p$
   overshoot, before any of this can be read for that discharge.